In [1]:
# directories
pdfs_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\pdfs'
ocr_text_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\ocr_text'
extracted_text_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\extracted_csv'

In [2]:
# 1871 - 1876 extraction
import csv
import re
from typing import Tuple

# Known US states and territories from that era
STATES = {
    "ALABAMA", "ARKANSAS", "ARIZONA", "CALIFORNIA", "COLORADO", "CONNECTICUT",
    "DELAWARE", "DISTRICT OF COLUMBIA", "FLORIDA", "GEORGIA", "IDAHO", "ILLINOIS",
    "INDIANA", "IOWA", "KANSAS", "KENTUCKY", "LOUISIANA", "MAINE", "MARYLAND",
    "MASSACHUSETTS", "MICHIGAN", "MINNESOTA", "MISSISSIPPI", "MISSOURI", "MONTANA",
    "NEBRASKA", "NEVADA", "NEW HAMPSHIRE", "NEW JERSEY", "NEW MEXICO", "NEW YORK",
    "NORTH CAROLINA", "OHIO", "OREGON", "PENNSYLVANIA", "RHODE ISLAND",
    "SOUTH CAROLINA", "TENNESSEE", "TEXAS", "UTAH", "VERMONT", "VIRGINIA",
    "WASHINGTON", "WEST VIRGINIA", "WISCONSIN", "WYOMING",
    "INDIAN TERRITORY", "DAKOTA", "DOMINION OF CANADA", "BRITISH COLONIES"
}


def extract_frequency(text: str) -> str:
    """Extract publication frequency."""
    freq_map = [
        (r'every\s*morning', 'Daily'),
        (r'every\s*evening', 'Daily'),
        (r'every\s*afternoon', 'Daily'),
        (r'every\s*day', 'Daily'),
        (r'semi-weekly', 'Semi-weekly'), (r'tri-weekly', 'Tri-weekly'),
        (r'bi-weekly', 'Bi-weekly'), (r'bi-monthly', 'Bi-monthly'),
        (r'semi-month', 'Semi-monthly'),
        (r'sundays?', 'Sundays'), (r'mondays?', 'Mondays'), (r'tuesdays?', 'Tuesdays'),
        (r'wednesdays?', 'Wednesdays'), (r'thursdays?', 'Thursdays'),
        (r'fridays?', 'Fridays'), (r'saturdays?', 'Saturdays'),
        (r'\bdaily\b', 'Daily'), (r'\bweekly\b', 'Weekly'),
        (r'\bmonthly\b', 'Monthly'), (r'\bquarterly\b', 'Quarterly'),
    ]
    text_lower = text.lower()
    for pattern, freq in freq_map:
        if re.search(pattern, text_lower):
            return freq
    return ""


def extract_political(text: str) -> str:
    """Extract political affiliation."""
    affil_map = [
        (r'\bdemocrat', 'Democratic'), (r'\brepublican', 'Republican'),
        (r'\bindependent', 'Independent'), (r'\bneutral', 'Neutral'),
        (r'\bliberal', 'Liberal'), (r'\bconservative', 'Conservative'),
        (r'\bgreenback', 'Greenback'), (r'\bprohibition', 'Prohibition'),
        (r'\bbaptist', 'Baptist'), (r'\bcongregational', 'Congregational'),
        (r'\bmethodist', 'Methodist'), (r'\buniversalist', 'Universalist'),
        (r'\breligious', 'Religious'), (r'\bagricultural', 'Agricultural'),
        (r'\bliterary', 'Literary'), (r'\bgerman', 'German'),
        (r'\bcomic', 'Comic'),
    ]
    text_lower = text.lower()
    for pattern, affil in affil_map:
        if re.search(pattern, text_lower):
            return affil
    return ""


def extract_price(text: str) -> str:
    """Extract subscription price."""
    patterns = [
        r'subscription\s*\$\s*(\d+)\s+(\d+)',
        r'subscription\s*\$\s*(\d+\.\d+)',
        r'subscription\s*\$\s*(\d+)',
    ]
    for pattern in patterns:
        match = re.search(pattern, text.lower())
        if match:
            groups = match.groups()
            if len(groups) == 2:
                return f"${groups[0]}.{groups[1]}"
            return f"${groups[0]}"
    return ""


def extract_established(text: str) -> str:
    """Extract year established."""
    patterns = [
        r'estab-?\s*lished\s*(\d{4})',
        r'established\s*(\d{4})',
        r're-established\s*(\d{4})',
    ]
    for pattern in patterns:
        match = re.search(pattern, text.lower())
        if match:
            return match.group(1)
    return ""


def extract_editor_publisher(text: str) -> Tuple[str, str]:
    """Extract editor and publisher names."""
    editor = ""
    publisher = ""
    
    # Pattern for "X, editor(s) and publisher(s)" (same person/people)
    match = re.search(r'([A-Z][A-Za-z\.\s\&\-,]+?),?\s*ed[-\s]*i[-\s]*t[-\s]*o[-\s]*r[-\s]*s?\s*and\s+pub[-\s]*l[-\s]*i[-\s]*s[-\s]*h[-\s]*e[-\s]*r[-\s]*s?', text)
    if match:
        name = re.sub(r'\s+', ' ', match.group(1).strip().rstrip(',;'))
        return name, name
    
    # Pattern for "X, editor(s) and proprietor(s)" (same person/people)
    match = re.search(r'([A-Z][A-Za-z\.\s\&\-,]+?),?\s*ed[-\s]*i[-\s]*t[-\s]*o[-\s]*r[-\s]*s?\s*and\s+pro[-\s]*p[-\s]*r[-\s]*i[-\s]*e[-\s]*t[-\s]*o[-\s]*r[-\s]*s?', text)
    if match:
        name = re.sub(r'\s+', ' ', match.group(1).strip().rstrip(',;'))
        return name, name
    
    # Pattern for editor
    match = re.search(r'([A-Z][A-Za-z\.\s\&\-,]+?),?\s*ed[-\s]*i[-\s]*t[-\s]*o[-\s]*r[-\s]*s?[;,:\s]', text)
    if match:
        editor = re.sub(r'\s+', ' ', match.group(1).strip().rstrip(',;'))
    
    # Pattern for publisher
    match = re.search(r'([A-Z][A-Za-z\.\s\&\-,]+(?:\s+Co\.)?),?\s*pub[-\s]*l[-\s]*i[-\s]*s[-\s]*h[-\s]*e[-\s]*r[-\s]*s?[;,:\s\.]', text)
    if match:
        publisher = re.sub(r'\s+', ' ', match.group(1).strip().rstrip(',;'))
    
    # Pattern for proprietor (extracted as publisher)
    if not publisher:
        match = re.search(r'([A-Z][A-Za-z\.\s\&\-,]+(?:\s+Co\.)?),?\s*pro[-\s]*p[-\s]*r[-\s]*i[-\s]*e[-\s]*t[-\s]*o[-\s]*r[-\s]*s?[;,:\s\.]', text)
        if match:
            publisher = re.sub(r'\s+', ' ', match.group(1).strip().rstrip(',;'))
    
    return editor, publisher


def extract_circulation(text: str) -> str:
    """Extract circulation number."""
    # Normalize line-break hyphens first
    text = re.sub(r'-\s+', '', text)
    
    # Pattern explanation:
    # - circ?(?:ulation|'n|e'n) : matches circulation, crculation, circ'n, cire'n
    # - [\s\-]* : optional whitespace or hyphens after the word
    # - (?:about|approximately|nearly|over|around)? : optional qualifier words
    # - [\s\-]* : optional whitespace or hyphens
    # - (?:\w+\s+)? : optional word like "daily" before the number
    # - (\d[\d,]*) : the circulation number
    pattern = r"circ?(?:ulation|'n|e'n)[\s\-]*(?:about|approximately|nearly|over|around)?[\s\-]*(?:\w+\s+)?(\d[\d,]*)"
    
    # Also check for "claims XXX" pattern
    claims_pattern = r"claims[\s\-]+(\d[\d,]*)"
    
    match = re.search(pattern, text.lower())
    if not match:
        match = re.search(claims_pattern, text.lower())
    
    if match:
        circ = match.group(1).replace(',', '')
        if 'estimated' in text.lower() or "est'd" in text.lower():
            return f"{circ} (estimated)"
        return circ
    return ""


def process_file(input_file, output_file):
    """Process a newspaper directory file and extract entries to CSV."""
    
    with open(input_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Remove page markers
    content = re.sub(r'---\s*Page\s+\d+\s*---', ' ', content)
    
    # Fix common OCR Greek letter substitutions (uppercase)
    content = content.replace('Α', 'A')  # Greek Alpha -> A
    content = content.replace('Β', 'B')  # Greek Beta -> B
    content = content.replace('Ε', 'E')  # Greek Epsilon -> E
    content = content.replace('Η', 'H')  # Greek Eta -> H
    content = content.replace('Ι', 'I')  # Greek Iota -> I
    content = content.replace('Κ', 'K')  # Greek Kappa -> K
    content = content.replace('Μ', 'M')  # Greek Mu -> M
    content = content.replace('Ν', 'N')  # Greek Nu -> N
    content = content.replace('Ο', 'O')  # Greek Omicron -> O
    content = content.replace('Ρ', 'P')  # Greek Rho -> P
    content = content.replace('Τ', 'T')  # Greek Tau -> T
    content = content.replace('Χ', 'X')  # Greek Chi -> X
    content = content.replace('Ζ', 'Z')  # Greek Zeta -> Z
    content = content.replace('Θ', 'O')  # Greek Theta -> O (visually similar)
    content = content.replace('Φ', 'O')  # Greek Phi -> O (visually similar)
    
    # Fix common OCR Cyrillic letter substitutions
    content = content.replace('С', 'C')  # Cyrillic Es -> C
    content = content.replace('О', 'O')  # Cyrillic O -> O
    content = content.replace('Р', 'P')  # Cyrillic Er -> P
    content = content.replace('Ф', 'O')  # Cyrillic Ef -> O
    content = content.replace('А', 'A')  # Cyrillic A -> A
    content = content.replace('Е', 'E')  # Cyrillic Ie -> E
    content = content.replace('Н', 'H')  # Cyrillic En -> H
    content = content.replace('В', 'B')  # Cyrillic Ve -> B
    content = content.replace('К', 'K')  # Cyrillic Ka -> K
    content = content.replace('М', 'M')  # Cyrillic Em -> M
    content = content.replace('Т', 'T')  # Cyrillic Te -> T
    
    # Fix lowercase Greek/Cyrillic
    content = content.replace('ο', 'o')  # Greek lowercase omicron -> o
    content = content.replace('а', 'a')  # Cyrillic lowercase a -> a
    content = content.replace('е', 'e')  # Cyrillic lowercase ie -> e
    content = content.replace('о', 'o')  # Cyrillic lowercase o -> o
    content = content.replace('р', 'p')  # Cyrillic lowercase er -> p
    content = content.replace('с', 'c')  # Cyrillic lowercase es -> c
    
    # Fix OCR diacritical errors
    content = content.replace('Ü', 'U')
    content = content.replace('Ö', 'O')
    content = content.replace('Ä', 'A')
    content = content.replace('É', 'E')
    content = content.replace('È', 'E')
    content = content.replace('Ñ', 'N')
    content = content.replace('Ç', 'C')
    
    # Remove OCR artifacts: sequences of O-like characters before town names
    # Matches patterns like "OOO ", "OOD ", "COO ", "CODO ", etc.
    content = re.sub(r'\b[OoCcDd0ΘΦ]{2,}\s+([A-Z]{2,})', r'\1', content)
    
    # Fix missing space between ALL CAPS town and Capitalized newspaper name
    # e.g., "VAN BURENArgus" -> "VAN BUREN Argus"
    content = re.sub(r'([A-Z]{4})([A-Z][a-z])', r'\1 \2', content)
    
    # Normalize whitespace
    text = ' '.join(content.split())
    
    results = []
    
    # Build state position index
    state_positions = []
    
    for state in STATES:
        # Pattern allows optional space before period: "ARKANSAS ." or "ARKANSAS."
        pattern = re.compile(r'\b' + re.escape(state) + r'\s*\.', re.IGNORECASE)
        for m in pattern.finditer(text):
            state_positions.append((m.start(), state))
    
    state_positions.sort()
    
    # Remove duplicate state entries at same/nearby positions
    filtered_positions = []
    for pos, state in state_positions:
        if not filtered_positions or pos - filtered_positions[-1][0] > 10:
            filtered_positions.append((pos, state))
    state_positions = filtered_positions
    
    # Main pattern for newspaper entries
    pattern = re.compile(
        r'\b'
        r'([A-Z][A-Z\'\-]+(?:\s+[A-Z][A-Z\'\-]+)*)'  # Group 1: Town (ALL CAPS words)
        r'\s*[,.\s]\s*'                              # Separator
        r'([A-Z][a-z][^;:†]*?)'                      # Group 2: Newspaper name
        r'\s*[;:†]'                                  # Delimiter
    )
    
    matches = list(pattern.finditer(text))
    
    # First pass: identify valid entries
    valid_matches = []
    for match in matches:
        pos = match.start()
        
        # Determine current state based on position
        match_state = None
        for sp, st in reversed(state_positions):
            if sp < pos:
                match_state = st
                break
        
        if not match_state:
            continue
        
        town = match.group(1).strip().rstrip(' ,.')
        newspaper = match.group(2).strip().rstrip(' ,.')
        
        # Skip index/header content
        if any(kw in newspaper.lower() for kw in ['list of', 'index', 'page']):
            continue
            
        # Skip if newspaper contains what looks like a page header
        if re.search(r'\b\d+\s+[A-Z]{4,}\.', newspaper):
            continue
        
        if len(town) >= 2 and len(newspaper) >= 2:
            valid_matches.append((match, match_state, town, newspaper))
    
    # Second pass: build results
    for i, (match, match_state, town, newspaper) in enumerate(valid_matches):
        if i + 1 < len(valid_matches):
            raw_text = text[match.start():valid_matches[i + 1][0].start()].strip()
        else:
            raw_text = text[match.start():].strip()
        
        # Extract additional fields from raw_text
        frequency = extract_frequency(raw_text)
        political = extract_political(raw_text)
        editor, publisher = extract_editor_publisher(raw_text)
        circulation = extract_circulation(raw_text)
        
        results.append({
            'state': match_state,
            'town': town.title(),
            'newspaper': newspaper,
            'frequency': frequency,
            'political': political,
            'editor': editor,
            'publisher': publisher,
            'circulation': circulation,
            'raw_text': raw_text
        })
    
    # Deduplicate
    seen = set()
    unique = []
    for r in results:
        key = (r['state'], r['town'], r['newspaper'])
        if key not in seen:
            seen.add(key)
            unique.append(r)
    
    # Remove known false positive entries from document header
    false_positives = {
        ("NEW YORK", "York", "January 1, 1869. ee ~~ CONTENTS"),
        ("NEW YORK", "Xiv", "Newspaper Directory Advertiser. XV. A circular to Advertisers, containing the names of more than one thousand newspapers, among which will be found the best advertising mediums in America"),
    }
    unique = [r for r in unique if (r['state'], r['town'], r['newspaper']) not in false_positives]
    
    print(f"{input_file}: {len(unique)} entries found")
    print(f"  States detected at positions: {state_positions[:10]}...")  # Debug
    
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['state', 'town', 'newspaper', 'frequency', 'political', 'editor', 'publisher', 'circulation', 'raw_text'])
        writer.writeheader()
        writer.writerows(unique)
    
    return unique

import os
for file in os.listdir(ocr_text_dir)[1:5]:
    input_file = os.path.join(ocr_text_dir, file)
    output_file = os.path.join(extracted_text_dir, file[:-3] + 'csv')
    results = process_file(input_file, output_file)

C:\Users\samwt\Documents\ExtendedEssay\data\raw\rowells_directories\ocr_text\Rowell 1871.txt: 5878 entries found
  States detected at positions: [(1645, 'ALABAMA'), (6202, 'ALABAMA'), (6936, 'ALABAMA'), (10647, 'ALABAMA'), (14436, 'ALABAMA'), (16217, 'ARKANSAS'), (17901, 'ARKANSAS'), (21657, 'ARKANSAS'), (25257, 'CALIFORNIA'), (28971, 'CALIFORNIA')]...
C:\Users\samwt\Documents\ExtendedEssay\data\raw\rowells_directories\ocr_text\Rowell 1872.txt: 6241 entries found
  States detected at positions: [(3031, 'ALABAMA'), (5793, 'ALABAMA'), (8365, 'ALABAMA'), (9568, 'ALABAMA'), (9807, 'ALABAMA'), (15944, 'ALABAMA'), (17629, 'ARKANSAS'), (19444, 'ARKANSAS'), (23221, 'ARKANSAS'), (26938, 'ARKANSAS')]...
C:\Users\samwt\Documents\ExtendedEssay\data\raw\rowells_directories\ocr_text\Rowell 1873.txt: 6550 entries found
  States detected at positions: [(4234, 'ALABAMA'), (6070, 'ALABAMA'), (8785, 'ALABAMA'), (9804, 'ALABAMA'), (16230, 'ALABAMA'), (18164, 'ARKANSAS'), (21031, 'ARKANSAS'), (23281, 'ARKA